# Microsoft Agent Framework — Lab 1: Basics

Welcome! Microsoft Agent Framework is the **direct successor to both AutoGen and Semantic Kernel**, combining AutoGen's simple agent abstractions with Semantic Kernel's enterprise features.

In this first lab we'll cover:
1. **Model Client** — connecting to OpenAI
2. **Agent** — creating and running an agent
3. **Tools** — giving the agent a function to call
4. **Streaming** — getting responses token-by-token

**Install:**
```bash
pip install agent-framework agent-framework-openai
```

In [11]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### Concept 1: The Model Client

The `OpenAIChatCompletionClient` wraps the OpenAI Chat Completions API.  
It reads `OPENAI_API_KEY` from the environment automatically.

In [12]:
from agent_framework.openai import OpenAIChatCompletionClient

# The client reads OPENAI_API_KEY from the environment
client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Client created:", client)

Client created: <agent_framework_openai._chat_completion_client.OpenAIChatCompletionClient object at 0x75beedb19a90>


### Concept 2: The Agent

Call `.as_agent()` on any chat client to get a standard `Agent` object.  
You can give it a `name` and `instructions` (the system prompt).

In [3]:
airline_agent = client.as_agent(
    name="airline_agent",
    instructions="You are a helpful assistant for an airline. You give short, humorous answers.",
)

result = await airline_agent.run("I'd like to go to London")
print(result)

Great choice! Just remember to practice your British “hello” and “cheerio” before you go. It’s like a free ticket to instant charm! 🚀✈️


### Concept 3: Streaming

Pass `stream=True` to get token-by-token output. Each chunk has a `.text` attribute.

In [4]:
print("Agent: ", end="", flush=True)
async for chunk in airline_agent.run("What are the best seats on a long-haul flight?", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

Agent: The best seats are usually the ones that don’t come with a crying baby or a snoring seatmate! But if you want to be strategic, aim for the exit row or window seat—great views and extra legroom! Just make sure you remember to (gasp) stretch and walk around—suddenly becoming a pretzel is not part of the fun!


### Concept 4: Tools

Let's build a local ticket price database and give the agent a tool to query it.  
Tools are just Python functions — annotate parameters and add a docstring.

In [5]:
import os
import sqlite3

DB_PATH = "tickets.db"

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
c = conn.cursor()
c.execute("CREATE TABLE cities (city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()

print("Database created")

Database created


In [6]:
def save_city_price(city_name: str, round_trip_price: float) -> None:
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        "REPLACE INTO cities (city_name, round_trip_price) VALUES (?, ?)",
        (city_name.lower(), round_trip_price),
    )
    conn.commit()
    conn.close()


save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Rome", 499)
save_city_price("Madrid", 550)
save_city_price("Barcelona", 580)
save_city_price("Berlin", 525)

print("Prices loaded")

Prices loaded


In [7]:
def get_city_price(city_name: str) -> float | None:
    """Get the round-trip ticket price to travel to a city."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        "SELECT round_trip_price FROM cities WHERE city_name = ?",
        (city_name.lower(),),
    )
    result = c.fetchone()
    conn.close()
    return result[0] if result else None


print(get_city_price("Rome"))  # 499

499.0


### Putting it all together — Agent with a Tool

Pass the tool function via the `tools` parameter. The framework automatically generates the JSON schema from the function signature and docstring.

In [8]:
smart_agent = client.as_agent(
    name="smart_airline_agent",
    instructions=(
        "You are a helpful assistant for an airline. "
        "You give short, humorous answers and always include the round-trip ticket price."
    ),
    tools=get_city_price,
)

result = await smart_agent.run("I'd like to go to London")
print(result)

A trip to London will cost you just $299! That's cheaper than a round of drinks at a pub there, and you'll still get a seat!


### Multiple tools

Pass a list to add several tools at once.

In [9]:
def list_all_cities() -> list[str]:
    """List all cities available in the airline ticket database."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT city_name FROM cities")
    rows = c.fetchall()
    conn.close()
    return [row[0].title() for row in rows]


multi_tool_agent = client.as_agent(
    name="multi_tool_agent",
    instructions=(
        "You are a helpful airline assistant. "
        "You can look up prices and list available destinations."
    ),
    tools=[get_city_price, list_all_cities],
)

result = await multi_tool_agent.run(
    "What destinations do you fly to, and what's the cheapest option?"
)
print(result)

Here are the destinations we fly to and their round-trip ticket prices:

1. **Barcelona** - $580
2. **Berlin** - $525
3. **London** - $299 (cheapest option)
4. **Madrid** - $550
5. **Paris** - $399
6. **Rome** - $499

The cheapest destination is **London** at **$299**.


### Using the `@tool` decorator (optional)

The `@tool` decorator from `agent_framework` gives you explicit control over the tool name and description shown to the model.

In [10]:
from agent_framework import tool


@tool
def get_ticket_price(city: str) -> str:
    """Look up the round-trip airfare to a destination city."""
    price = get_city_price(city)
    if price is None:
        return f"Sorry, we don't fly to {city}."
    return f"Round-trip to {city.title()}: ${price:.0f}"


decorated_agent = client.as_agent(
    name="decorated_agent",
    instructions="You are a concise airline assistant.",
    tools=get_ticket_price,
)

result = await decorated_agent.run("How much is a round trip to Barcelona?")
print(result)

A round trip to Barcelona costs $580.


---
### Summary

| Concept | Class / Method |
|---|---|
| Model client | `OpenAIChatCompletionClient(model=...)` |
| Create agent | `client.as_agent(name=..., instructions=..., tools=...)` |
| Run agent | `await agent.run("...")` |
| Stream agent | `async for chunk in agent.run("...", stream=True)` |
| Tool decorator | `@tool` from `agent_framework` |

In Lab 2 we'll look at structured outputs, multi-turn sessions, and coordinating multiple agents.